# 17 · Calculus channels for limit-order-book signals

On the FI-2010 / DeepLOB mid-price movement task, omnibias adds **calculus
channels**: local Taylor jets (velocity, curvature, higher derivatives) of the
mid-price, spread, and order-flow imbalance. These closed-form temporal features
augment a standard LOB feature set. This notebook shows the feature transform on
a synthetic order book (data-free); the full FI-2010 benchmark is in
`examples/symbolic_discovery/financial_signal_discovery/`.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "..")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from examples.symbolic_discovery.financial_signal_discovery.benchmark import build_fi2010_calculus_features

## A synthetic order book

Columns are `[ask_price, ask_size, bid_price, bid_size]`. The mid-price follows a
drifting random walk; sizes fluctuate.

In [ ]:
rng = np.random.default_rng(1)
n = 600
mid = 100.0 + np.cumsum(rng.normal(0.0, 0.02, size=n)) + 0.5 * np.sin(np.linspace(0, 6 * np.pi, n))
half_spread = 0.01 + 0.002 * rng.random(n)
ask_p = mid + half_spread
bid_p = mid - half_spread
ask_sz = 100 + 20 * rng.standard_normal(n)
bid_sz = 100 + 20 * rng.standard_normal(n)
lob = np.column_stack([ask_p, ask_sz, bid_p, bid_sz])
feats = build_fi2010_calculus_features(lob, lookback=20)
print("calculus feature block shape:", feats.shape, "(9 jets: mid/spread/imbalance x velocity/curvature/4th)")

## Mid-price velocity and curvature

The first two channels are the closed-form local velocity and curvature of the
mid-price — momentum and acceleration signals derived analytically, not by noisy
finite differencing.

In [ ]:
vel = feats[:, 0]
curv = feats[:, 1]
t = np.arange(n)
fig, (axl, axr) = plt.subplots(2, 1, figsize=(8.6, 5.6), sharex=True)
axl.plot(t, mid, color=PRIMARY)
axl.set_title("Synthetic mid-price"); axl.set_ylabel("price")
axr.plot(t, vel, color=ACCENT, label="velocity (1st jet)")
axr.plot(t, curv, color=GOOD, lw=1.5, label="curvature (2nd jet)")
axr.axhline(0, color="#888", lw=1)
axr.set_title("Closed-form mid-price calculus channels"); axr.set_xlabel("tick"); axr.legend()
plt.tight_layout()

## Takeaway

Local Taylor jets give smooth momentum/acceleration features for free. On real
FI-2010 these calculus channels improve a DeepLOB-style classifier over the raw
LOB baseline — run the `evaluate_milestone*` functions in the experiment package
after fetching the dataset.